In [ ]:
!pip install faiss-cpu

In [2]:
import pandas as pd

df = pd.read_excel("/kaggle/input/datasets/hamnainam/songlyrics/lyriics.xlsx")
print(df.shape)
print(df.columns)
print(df.head(3))

# Check lyrics length distribution
df['lyric_length'] = df['Lyrics'].apply(len)
print(df['lyric_length'].describe())

(147, 4)
Index(['Index', 'Album', 'Song Name', 'Lyrics'], dtype='object')
   Index         Album                  Song Name  \
0      0  Taylor Swift  Mary's Song (Oh My My My)   
1      1  Taylor Swift     A Perfectly Good Heart   
2      2  Taylor Swift                 Tim McGraw   

                                              Lyrics  
0  She said I was seven and you were nine I looke...  
1  Why would you wanna break A perfectly good hea...  
2  He said the way my blue eyes shined Put those ...  
count     147.000000
mean     1851.863946
std       449.538540
min       811.000000
25%      1540.000000
50%      1818.000000
75%      2121.500000
max      3532.000000
Name: lyric_length, dtype: float64


## Using an encoder with a classification head on top

In [3]:
from transformers import pipeline

emotion_classifier = pipeline("text-classification", 
                               model="j-hartmann/emotion-english-distilroberta-base",
                               top_k=None)

# Test on a single song first
sample = df['Lyrics'][0]
print(f"Song: {df['Song Name'][0]}")
print(f"Char length: {len(sample)}")
result = emotion_classifier(sample)
print(result)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Song: Mary's Song (Oh My My My)
Char length: 1602
[[{'label': 'fear', 'score': 0.4063781797885895}, {'label': 'surprise', 'score': 0.23436608910560608}, {'label': 'neutral', 'score': 0.11134731769561768}, {'label': 'anger', 'score': 0.10027838498353958}, {'label': 'joy', 'score': 0.08611677587032318}, {'label': 'sadness', 'score': 0.03833812475204468}, {'label': 'disgust', 'score': 0.023175150156021118}]]


# See the effect of preprocessing

In [4]:
import re

def preprocess(text):
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def get_emotion(text):
    result = emotion_classifier(text)[0]
    return {r['label']: round(r['score'], 3) for r in result}

for i in range(10):
    song = df['Song Name'][i]
    raw = df['Lyrics'][i]
    clean = preprocess(raw)
    
    raw_result = get_emotion(raw)
    clean_result = get_emotion(clean)
    
    print(f"\n--- {song} ---")
    print(f"Raw:   {raw_result}")
    print(f"Clean: {clean_result}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



--- Mary's Song (Oh My My My) ---
Raw:   {'fear': 0.406, 'surprise': 0.234, 'neutral': 0.111, 'anger': 0.1, 'joy': 0.086, 'sadness': 0.038, 'disgust': 0.023}
Clean: {'surprise': 0.455, 'joy': 0.132, 'fear': 0.125, 'neutral': 0.102, 'anger': 0.095, 'sadness': 0.057, 'disgust': 0.033}

--- A Perfectly Good Heart ---
Raw:   {'sadness': 0.397, 'fear': 0.347, 'surprise': 0.112, 'anger': 0.104, 'neutral': 0.028, 'disgust': 0.008, 'joy': 0.005}
Clean: {'sadness': 0.397, 'fear': 0.216, 'anger': 0.173, 'surprise': 0.169, 'neutral': 0.026, 'disgust': 0.015, 'joy': 0.005}

--- Tim McGraw ---
Raw:   {'sadness': 0.607, 'joy': 0.14, 'neutral': 0.132, 'surprise': 0.08, 'fear': 0.028, 'anger': 0.008, 'disgust': 0.005}
Clean: {'sadness': 0.665, 'surprise': 0.12, 'neutral': 0.101, 'joy': 0.098, 'fear': 0.007, 'disgust': 0.005, 'anger': 0.004}

--- Teardrops On My Guitar ---
Raw:   {'surprise': 0.873, 'neutral': 0.046, 'joy': 0.028, 'sadness': 0.02, 'fear': 0.019, 'anger': 0.012, 'disgust': 0.002}
Clean

# No chunking returns an error as the model can only process 512 tokens

In [5]:
'''from tqdm import tqdm

tqdm.pandas()

def get_emotion_vector(lyrics):
    clean = preprocess(lyrics)
    result = emotion_classifier(clean)[0]
    return {r['label']: round(r['score'], 3) for r in result}

df['emotion_vector'] = df['Lyrics'].progress_apply(get_emotion_vector)

print(df[['Song Name', 'emotion_vector']].to_string())
'''

"from tqdm import tqdm\n\ntqdm.pandas()\n\ndef get_emotion_vector(lyrics):\n    clean = preprocess(lyrics)\n    result = emotion_classifier(clean)[0]\n    return {r['label']: round(r['score'], 3) for r in result}\n\ndf['emotion_vector'] = df['Lyrics'].progress_apply(get_emotion_vector)\n\nprint(df[['Song Name', 'emotion_vector']].to_string())\n"

 # Chuking + Cleaning

In [6]:
from tqdm.notebook import tqdm
tqdm.pandas()

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("j-hartmann/emotion-english-distilroberta-base")

def get_emotion_vector(lyrics):
    clean = preprocess(lyrics)
    tokens = tokenizer.encode(clean, add_special_tokens=False)
    
    # Split into chunks of 500 tokens (leaving room for special tokens)
    chunk_size = 500
    token_chunks = [tokens[i:i+chunk_size] for i in range(0, len(tokens), chunk_size)]
    
    all_scores = []
    for chunk in token_chunks:
        text = tokenizer.decode(chunk)
        result = emotion_classifier(text)[0]
        all_scores.append({r['label']: r['score'] for r in result})
    
    avg = {label: round(sum(s[label] for s in all_scores) / len(all_scores), 3)
           for label in all_scores[0]}
    return avg

df['emotion_vector'] = df['Lyrics'].progress_apply(get_emotion_vector)
print(df[['Song Name', 'emotion_vector']].to_string())

  0%|          | 0/147 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (637 > 512). Running this sequence through the model will result in indexing errors


                                   Song Name                                                                                                          emotion_vector
0                  Mary's Song (Oh My My My)  {'surprise': 0.455, 'joy': 0.132, 'fear': 0.125, 'neutral': 0.102, 'anger': 0.095, 'sadness': 0.057, 'disgust': 0.033}
1                     A Perfectly Good Heart  {'sadness': 0.397, 'fear': 0.216, 'anger': 0.173, 'surprise': 0.169, 'neutral': 0.026, 'disgust': 0.015, 'joy': 0.005}
2                                 Tim McGraw   {'sadness': 0.665, 'surprise': 0.12, 'neutral': 0.101, 'joy': 0.098, 'fear': 0.007, 'disgust': 0.005, 'anger': 0.004}
3                     Teardrops On My Guitar  {'surprise': 0.943, 'neutral': 0.019, 'joy': 0.015, 'anger': 0.009, 'sadness': 0.007, 'fear': 0.004, 'disgust': 0.003}
4                                Cold as You  {'sadness': 0.538, 'surprise': 0.155, 'disgust': 0.103, 'neutral': 0.087, 'fear': 0.053, 'anger': 0.043, 'joy': 0.022}
5         

In [8]:
!pip install sentence-transformers

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all lyrics
df['embedding'] = df['Lyrics'].apply(lambda x: model.encode(preprocess(x)))

print(f"Embedding shape: {df['embedding'][0].shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (384,)


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
def recommend(user_input, top_k=3):
    user_embedding = model.encode(preprocess(user_input)).reshape(1, -1)
    song_embeddings = np.stack(df['embedding'].values)
    scores = cosine_similarity(user_embedding, song_embeddings)[0]
    top_indices = scores.argsort()[::-1][:top_k]
    
    for i in top_indices:
        print(f"{df['Song Name'][i]} | score: {scores[i]:.3f}")

recommend("I feel heartbroken and miss someone I lost")
print("---")
recommend("I'm feeling angry and betrayed by a friend")
print("---")
recommend("I feel nostalgic and happy")

Sad Beautiful Tragic | score: 0.392
How You Get The Girl | score: 0.383
Hey Stephen | score: 0.365
---
This Is Why We Can't Have Nice Things | score: 0.370
Better Than Revenge | score: 0.366
I Did Something Bad | score: 0.332
---
happiness | score: 0.346
22 | score: 0.299
Back to December | score: 0.292


In [12]:
model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

# Re-embed everything
df['embedding'] = df['Lyrics'].apply(lambda x: model.encode(preprocess(x)))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
def recommend(user_input, top_k=3):
    user_embedding = model.encode(preprocess(user_input)).reshape(1, -1)
    song_embeddings = np.stack(df['embedding'].values)
    scores = cosine_similarity(user_embedding, song_embeddings)[0]
    top_indices = scores.argsort()[::-1][:top_k]
    
    for i in top_indices:
        print(f"{df['Song Name'][i]} | score: {scores[i]:.3f}")

recommend("I feel heartbroken and miss someone I lost")
print("---")
recommend("I'm feeling angry and betrayed by a friend")
print("---")
recommend("I feel nostalgic and happy")

The Story Of Us | score: 0.435
Cold as You | score: 0.424
Breathe | score: 0.422
---
The Way I Loved You | score: 0.385
Better Than Revenge | score: 0.372
Karma | score: 0.365
---
happiness | score: 0.471
Back to December | score: 0.469
The Best Day | score: 0.444


In [14]:
models_to_try = [
    'all-mpnet-base-v2',
    'multi-qa-mpnet-base-dot-v1',
    'all-roberta-large-v1',
    'msmarco-roberta-base-v3',
    'sentence-t5-large'
]

test_queries = [
    "I feel heartbroken and miss someone I lost",
    "I'm feeling angry and betrayed by a friend",
    "I feel nostalgic and happy",
    "I want to shake it off and not care what people think",
"I'm in a new relationship and everything feels magical"
]

for model_name in models_to_try:
    print(f"\n{'='*20} {model_name} {'='*20}")
    m = SentenceTransformer(model_name)
    embeddings = np.stack([m.encode(preprocess(l)) for l in df['Lyrics']])
    
    for query in test_queries:
        query_emb = m.encode(preprocess(query)).reshape(1, -1)
        scores = cosine_similarity(query_emb, embeddings)[0]
        top = scores.argsort()[::-1][:3]
        print(f"\n{query}")
        for i in top:
            print(f"  {df['Song Name'][i]} | {scores[i]:.3f}")


==================== all-mpnet-base-v2 ====================


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/3.15M [00:00<?, ?B/s]


I feel heartbroken and miss someone I lost
  evermore | 0.791
  coney island | 0.791
  Back to December | 0.788

I'm feeling angry and betrayed by a friend
  This Is Why We Can't Have Nice Things | 0.798
  Maroon | 0.792
  illicit affairs | 0.782

I feel nostalgic and happy
  All Too Well | 0.773
  The Best Day | 0.768
  Holy Ground | 0.765

I want to shake it off and not care what people think
  illicit affairs | 0.790
  Cold as You | 0.785
  The Man | 0.782

I'm in a new relationship and everything feels magical
  The Way I Loved You | 0.760
  Paper Rings | 0.749
  22 | 0.748


In [15]:
models_to_try = [
    'BAAI/bge-large-en-v1.5',
    'Qwen/Qwen3-Embedding-0.6B'
]
test_queries = [
    "I feel heartbroken and miss someone I lost",
    "I'm feeling angry and betrayed by a friend",
    "I feel nostalgic and happy",
    "I want to shake it off and not care what people think",
"I'm in a new relationship and everything feels magical"
]

for model_name in models_to_try:
    print(f"\n{'='*20} {model_name} {'='*20}")
    m = SentenceTransformer(model_name)
    embeddings = np.stack([m.encode(preprocess(l)) for l in df['Lyrics']])
    
    for query in test_queries:
        query_emb = m.encode(preprocess(query)).reshape(1, -1)
        scores = cosine_similarity(query_emb, embeddings)[0]
        top = scores.argsort()[::-1][:3]
        print(f"\n{query}")
        for i in top:
            print(f"  {df['Song Name'][i]} | {scores[i]:.3f}")


==================== BAAI/bge-large-en-v1.5 ====================


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]


I feel heartbroken and miss someone I lost
  coney island | 0.585
  Back to December | 0.553
  Breathe | 0.535

I'm feeling angry and betrayed by a friend
  betty | 0.470
  The Way I Loved You | 0.454
  This Is Why We Can't Have Nice Things | 0.432

I feel nostalgic and happy
  Mary's Song (Oh My My My) | 0.455
  The Best Day | 0.452
  Out Of The Woods | 0.441

I want to shake it off and not care what people think
  Shake It Off | 0.558
  You Need To Calm Down | 0.484
  The Man | 0.459

I'm in a new relationship and everything feels magical
  Enchanted | 0.492
  Everything Has Changed | 0.456
  The Way I Loved You | 0.445


# Qwen embeddings win

In [16]:
df['clean_lyrics'] = df['Lyrics'].apply(preprocess)

# Google Flan t5

In [17]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

flan_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
flan_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")

def summarize_lyrics(lyrics):
    prompt = f"Describe the emotional tone and themes of this song in 2-3 sentences, focusing on how it would make a listener feel:\n\n{lyrics[:1000]}"
    inputs = flan_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = flan_model.generate(inputs["input_ids"], max_length=100, min_length=30)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test on a few songs
for i in [0, 10, 51]:  # Mary's Song, Our Song, Red
    print(f"\n{df['Song Name'][i]}")
    print(summarize_lyrics(df['clean_lyrics'][i]))

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Mary's Song (Oh My My My)
ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki ki 

Stay Beautiful
Corey's eyes are like a jungle He smiles, it's like the radio He whispers songs into my window In words that nobody knows There's pretty girls on every corner They watch him as he's walking home Saying, "Does he know?" Will you ever know? You're beautiful, every little piece, love Don't you know you're really gonna be someone? Ask anyone And when you find everything you looked

Red
The song is about a man who is in love with a woman. The song is about a man who is in love with a woman.


In [18]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

phi_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
phi_model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct", 
                                                   torch_dtype=torch.float16,
                                                   device_map="auto")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [19]:
def summarize_lyrics(lyrics):
    prompt = f"<|user|>\nBased ONLY on the lyrics provided below, in 3 sentences describe: (1) what specific situation or story this song tells, and (2) what emotions it expresses. Do not add any details not present in the lyrics. Use specific emotion words like sadness, anger, joy, longing, heartbreak, nostalgia, betrayal, excitement, rage, happiness:\n\n{lyrics[:2000]}<|end|>\n<|assistant|>"    
    inputs = phi_tokenizer(prompt, return_tensors="pt").to(phi_model.device)
    outputs = phi_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    decoded = phi_tokenizer.decode(outputs[0], skip_special_tokens=False)
    summary = decoded.split("<|assistant|>")[-1].strip()
    summary = summary.replace("<|end|>", "").strip()
    return summary

for i in [28, 59, 51, 61, 105,117 , 52, 53, 123]:  # Mary's Song, Our Song, Red, 22, Shake It Off
    print(f"\n{df['Song Name'][i]}")
    print(summarize_lyrics(df['clean_lyrics'][i]))


Back to December
The song tells the story of a person who has experienced a painful breakup and is now reflecting on their past relationship with their ex-partner. The lyrics express feelings of regret, longing, and sadness as the singer acknowledges their mistakes and wishes they had realized the value of their relationship before it ended. The emotions conveyed include heartbreak, nostalgia, and a desire for reconciliation, as the singer contemplates the possibility of rekindling their love.

I Know Places
The song tells the story of a couple who are constantly on the run from their pursuers, who are depicted as hunters. The lyrics express a sense of fear, determination, and resilience as they navigate through dangerous situations, always staying one step ahead of their pursuers. The emotions conveyed include anxiety, excitement, and a deep sense of love and commitment to each other.

Red
The song tells the story of a passionate and intense love that ends abruptly, leaving the singe

In [20]:
df['summary'] = df['clean_lyrics'].progress_apply(summarize_lyrics)
print(df[['Song Name', 'summary']].to_string())

  0%|          | 0/147 [00:00<?, ?it/s]

                                   Song Name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     summary
0                  Mary's Song (Oh My My My)                                                                                                                                                                                                                                                                                   

In [21]:
print(summarize_lyrics(df['clean_lyrics'][50]))

The song tells the story of a past relationship where the narrator and their partner shared a deep connection, symbolized by the scarf left at the partner's sister's house. The emotions expressed include nostalgia, longing, and heartbreak, as the narrator reminisces about their time together, the innocence they shared, and the pain of the relationship's end. The song also conveys a sense of betrayal and sadness, as the partner keeps the scarf as a reminder of their past, despite the narrator's desire to move on and find their old self again.


In [22]:
df.to_excel("taylor_swift_with_summaries.xlsx", index=False)

In [23]:
df = df.drop(columns=['embedding'])

In [28]:
import faiss
import numpy as np

qwen_model = SentenceTransformer('Qwen/Qwen3-Embedding-0.6B')
df['summary_embedding'] = df['summary'].apply(lambda x: qwen_model.encode(x))

# Get embeddings as a numpy array
embeddings = np.stack(df['summary_embedding'].values).astype('float32')

# Normalize for cosine similarity
faiss.normalize_L2(embeddings)

# Build index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product = cosine similarity after normalization
index.add(embeddings)

print(f"Index built with {index.ntotal} vectors of dimension {dimension}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Index built with 147 vectors of dimension 1024


In [29]:
def recommend(user_input, top_k=3):
    user_embedding = qwen_model.encode(user_input).astype('float32').reshape(1, -1)
    faiss.normalize_L2(user_embedding)
    
    scores, indices = index.search(user_embedding, top_k)
    
    for score, idx in zip(scores[0], indices[0]):
        print(f"{df['Song Name'][idx]} | {score:.3f}")

recommend("I feel heartbroken and miss someone I lost")
print("---")
recommend("I'm feeling angry and betrayed by a friend")
print("---")
recommend("I feel nostalgic and happy")
print("---")
recommend("I want to shake it off and not care what people think")
print("---")
recommend("I'm in a new relationship and everything feels magical")

coney island | 0.582
my tears ricochet | 0.582
Breathe | 0.576
---
mad woman | 0.589
This Is Why We Can't Have Nice Things | 0.568
You Need To Calm Down | 0.566
---
It's Nice to Have a Friend | 0.482
Mary's Song (Oh My My My) | 0.478
tis the damn season | 0.459
---
Shake It Off | 0.486
The Lucky One | 0.430
Sweet Nothing | 0.419
---
Begin Again | 0.499
Out Of The Woods | 0.490
Sparks Fly | 0.458


In [30]:
df.to_excel("taylor_swift_with_summaries.xlsx", index=False)